# NB2 — Train M1 (XGBoost)

v7 pipeline. Runs on Kaggle GPU/CPU (XGBoost with `tree_method="hist"`, GPU optional).

M1 is the deterministic baseline: no uncertainty quantification. `epistemic_unc` is fixed at 0
by construction, not estimated with a proxy. The previous version of this notebook used a
Random-Forest predictive-variance proxy to attach an uncertainty score to M1 — that proxy is
removed here. It measured tree-to-tree disagreement in a *different* model (RF), not any
property of the XGBoost model actually being explained, so it did not have a defensible
theoretical grounding. M1's role in this study is a pure "no-UQ" reference point; giving it a
proxy epistemic score would blur that role.

Faithfulness is computed in three parallel variants (see NB5 for the same pattern applied to the
neural models):

- `prob`  — Comprehensiveness/Sufficiency in probability space (DeYoung et al., 2020), as before.
- `logit` — same masking procedure, effect measured in logit space. Not bounded above by `p_bar`,
  which avoids the ceiling-effect confound identified during the v6 audit.
- `norm`  — probability-space effect normalized by the "null difference"
  `p(x) - p(x_baseline)` (Carton, Rathore & Tan, 2020), which controls for the fact that
  comprehensiveness scores are not comparable in absolute magnitude across models or datasets.

Random-K control values are stored **per sample**, not only as a mean, so NB6 can test whether
the SHAP/LIME masking curve is steeper than the random-masking curve specifically in the
high-uncertainty stratum (this is the check the v6 design promised but never actually ran).

Outputs go to `/kaggle/working`. Upload the `NB2-outputs` folder split into `NB2-result`
(json/csv) and `NB2-models` (pkl) as described in the local directory layout.


## 1. Installs & imports

In [ ]:
!pip install xgboost shap lime fastparquet scikit-learn -q

import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import lime
import lime.lime_tabular
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import kendalltau

import warnings
warnings.filterwarnings("ignore")
print("Imports completed.")


## 2. Configuration

In [ ]:
class Config:
    INPUT_DIR = "/kaggle/input/xai-credit-preprocessed"
    OUTPUT_DIR = "/kaggle/working"
    SEED = 42
    DATASETS = ["home_credit", "taiwan", "gmsc"]
    # v7 audit fix: DATASET_FEATURES used to be a hard-coded dict here (with a stale value
    # for home_credit -- 122, not the real 105 numeric features once SK_ID_CURR and the
    # object-dtype columns are excluded), so the adaptive-K grid was silently computed from
    # the wrong denominator. get_k_vals now takes the *actual* feature count of the data
    # being processed instead of looking it up.

    SHAP_NSAMPLES = 500
    LIME_NSAMPLES = 500
    LIME_BACKGROUND_SIZE = 100

    STABILITY_N_INSTANCES = 20     # instances used for the reproducibility check
    STABILITY_N_RERUNS = 10        # explainer re-runs per instance
    SENSITIVITY_N_PERTURB = 10     # perturbations per instance for local-robustness
    SENSITIVITY_RADIUS = 0.01

    RANDOM_K_REPEATS = 10          # repeats of the random-K control, averaged per sample
    FAITH_CHUNK = 64          # so mau gop vao mot lan goi predict khi tinh
                              # faithfulness; chi anh huong toc do, khong doi ket qua

    @staticmethod
    def get_k_vals(n_features):
        percentages = [0.1, 0.2, 0.3, 0.4, 0.5]
        k_vals = [max(1, int(round(p * n_features))) for p in percentages]
        return sorted(set(k_vals))

    if not os.path.exists(INPUT_DIR):
        print(f"WARNING: {INPUT_DIR} not found. Assuming local test run.")
        INPUT_DIR = "../kaggle_outputs/xai-credit-preprocessed"
        os.makedirs(INPUT_DIR, exist_ok=True)
        os.makedirs(OUTPUT_DIR, exist_ok=True)


import random
random.seed(Config.SEED)
np.random.seed(Config.SEED)

## 3. Core metric functions

### Calibration
Standard ECE (equal-width binning), NLL, Brier score.

### Faithfulness — three variants
`calculate_faithfulness` masks the top-K features (ranked by |attribution|) with the train-set
mean and measures the resulting change in the model's output. It returns per-sample arrays
(`*_samples`) as well as means, and it does the same for a random-K control, repeated
`RANDOM_K_REPEATS` times and averaged per sample to reduce control-baseline variance.

- Comprehensiveness: `f(x) - f(x_masked)`   (mask the top-K important features out)
- Sufficiency:       `f(x) - f(x_only_topK)` (keep only the top-K features, mask everything else)

where `f` is either the raw probability (`prob`), the logit (`logit`), or the null-difference
normalized probability effect (`norm`). Sign is **not** clipped to non-negative (unlike the
v6 version of this function) — a negative Sufficiency score is a meaningful outcome (keeping the
top-K features raised the prediction above the original), and clipping it away discards that
information.


In [ ]:
def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob > bins[i]) & (y_prob <= bins[i + 1])
        if mask.sum() == 0:
            continue
        avg_conf = y_prob[mask].mean()
        avg_acc = y_true[mask].mean()
        ece += mask.sum() / len(y_true) * abs(avg_conf - avg_acc)
    return float(ece)


def compute_nll(y_true, y_prob, eps=1e-10):
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return float(-np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob)))


def compute_brier(y_true, y_prob):
    return float(np.mean((y_prob - y_true) ** 2))


def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def calculate_max_sensitivity(explain_fn, X, n_perturbations=10, radius=0.01, seed=0):
    """Local-robustness metric of Alvarez-Melis & Jaakkola (2018): the largest change in the
    explanation under a small input perturbation, normalized by the explanation's own norm.
    Distinct from the reproducibility metric below, which re-runs the explainer on the SAME
    input and measures estimator variance rather than input sensitivity."""
    rng = np.random.RandomState(seed)
    base = explain_fn(X)
    sens = np.zeros(len(X))
    for i in range(len(X)):
        x0 = X[i:i + 1]
        pert = x0 + rng.uniform(-radius, radius, size=(n_perturbations, X.shape[1]))
        pert_vals = explain_fn(pert)
        diffs = np.linalg.norm(pert_vals - base[i], axis=1)
        sens[i] = np.max(diffs) / (np.linalg.norm(base[i]) + 1e-9)
    return sens


def calculate_faithfulness(predict_fn, X, base_probs, attributions, k_list, baseline_matrix,
                            variants=("prob", "logit", "norm"), random_repeats=1, seed=42,
                            norm_tol=0.005, chunk_size=64):
    """Comprehensiveness / Sufficiency + the Random-K control, for ALL variants in one pass.

    Returns {variant: (comp, suff, comp_rand, suff_rand)}, each a {k: array(n_samples)} dict.

    baseline_matrix : (n, d) array -- ONE masking point per sample, not a single population
        mean. v7 audit: masking every sample to the SAME mean vector on correlated tabular
        features is itself an out-of-manifold point; how far a sample sits from it turns out
        to correlate with epistemic uncertainty and manufactures a spurious link between
        uncertainty and comprehensiveness that has nothing to do with explanation quality
        (baseline-swap experiment, answers FB3). Per-sample points drawn from the training
        distribution do not have this problem. Fixed once per dataset by the caller, so every
        model/method/variant/K comparison in this notebook sees the same draw.
    norm_tol : the 'norm' variant divides by (base_probs - p_null); when that is close to zero
        the ratio is undefined and is written as NaN rather than clamped to a floor -- the old
        floor (max(., 1e-6)) silently kept the sign of the numerator, which flips ~45-70% of
        samples on Home Credit / GMSC (p_null sits mid-distribution there) so that a BETTER
        explanation scores as MORE negative. See table10_metric_orientation in NB6.
    variants / chunk_size : v7 audit fix for RUNTIME, not for correctness. The previous version
        took a single `variant` and was called three times, but all three variants are different
        arithmetic on the SAME masked probabilities -- so two thirds of the masking work was
        thrown away. It also called predict_fn once per masked row; with M4 averaging 50 basins
        that is 50 single-row GPU launches per call, and at 2000 analysis samples the stage did
        not fit in a Kaggle session. Now every masked row for `chunk_size` samples goes through
        predict_fn in one batch. Numerically safe because every basin runs under .eval(), so
        BatchNorm uses running statistics and batch size cannot change a row's output; verified
        against the old implementation at float64 (max deviation 3.7e-14, NaN masks identical).
    """
    n, n_features = X.shape
    variants = tuple(variants)
    out = {v: tuple({k: np.full(n, np.nan) for k in k_list} for _ in range(4))
           for v in variants}

    p_null_local = predict_fn(baseline_matrix)
    null_diff = base_probs - p_null_local              # SIGNED, not clamped
    null_defined = np.abs(null_diff) > norm_tol
    base_logit = to_logit(base_probs)

    n_per_sample = len(k_list) * 2 * (1 + random_repeats)

    for start in range(0, n, chunk_size):
        stop = min(start + chunk_size, n)
        rows, meta = [], []
        for i in range(start, stop):
            x_base = X[i]
            baseline_row = baseline_matrix[i]
            top_idx = np.argsort(-np.abs(attributions[i]))
            for k in k_list:
                top_k = top_idx[:k]
                x_comp = x_base.copy()
                x_comp[top_k] = baseline_row[top_k]
                x_suff = baseline_row.copy()
                x_suff[top_k] = x_base[top_k]
                rows += [x_comp, x_suff]
                meta += [(i, k, "comp"), (i, k, "suff")]
                for r in range(random_repeats):
                    # same RNG stream as the pre-batching version: a fresh RandomState per
                    # (sample, repeat), so the random masks are unchanged and stay shared
                    # across variants, methods and model groups
                    rng = np.random.RandomState(seed + i * 1000 + r)
                    rand_idx = rng.choice(n_features, size=min(k, n_features), replace=False)
                    x_comp_r = x_base.copy()
                    x_comp_r[rand_idx] = baseline_row[rand_idx]
                    x_suff_r = baseline_row.copy()
                    x_suff_r[rand_idx] = x_base[rand_idx]
                    rows += [x_comp_r, x_suff_r]
                    meta += [(i, k, "comp_r"), (i, k, "suff_r")]

        probs = predict_fn(np.asarray(rows, dtype=X.dtype))
        assert len(probs) == len(meta) == (stop - start) * n_per_sample, "batch bookkeeping"

        acc = {}
        for key, p in zip(meta, probs):
            acc.setdefault(key, []).append(p)

        for i in range(start, stop):
            for k in k_list:
                p_comp = acc[(i, k, "comp")][0]
                p_suff = acc[(i, k, "suff")][0]
                pc_r = np.asarray(acc[(i, k, "comp_r")])
                ps_r = np.asarray(acc[(i, k, "suff_r")])
                for v in variants:
                    comp, suff, comp_rand, suff_rand = out[v]
                    if v == "prob":
                        comp[k][i] = base_probs[i] - p_comp
                        suff[k][i] = base_probs[i] - p_suff
                        comp_rand[k][i] = np.mean(base_probs[i] - pc_r)
                        suff_rand[k][i] = np.mean(base_probs[i] - ps_r)
                    elif v == "logit":
                        comp[k][i] = base_logit[i] - to_logit(np.array([p_comp]))[0]
                        suff[k][i] = base_logit[i] - to_logit(np.array([p_suff]))[0]
                        comp_rand[k][i] = np.mean([base_logit[i] - to_logit(np.array([q]))[0]
                                                   for q in pc_r])
                        suff_rand[k][i] = np.mean([base_logit[i] - to_logit(np.array([q]))[0]
                                                   for q in ps_r])
                    elif v == "norm":
                        if not null_defined[i]:
                            continue   # stays NaN at every K, same as the pre-batching version
                        comp[k][i] = (base_probs[i] - p_comp) / null_diff[i]
                        suff[k][i] = (base_probs[i] - p_suff) / null_diff[i]
                        comp_rand[k][i] = np.mean((base_probs[i] - pc_r) / null_diff[i])
                        suff_rand[k][i] = np.mean((base_probs[i] - ps_r) / null_diff[i])

    return out
def kendalls_w(shap_vals, lime_vals):
    """Per-sample Kendall's tau between |SHAP| and |LIME| feature rankings — agreement between
    the two explainers, not a faithfulness metric."""
    scores = np.zeros(len(shap_vals))
    for i in range(len(shap_vals)):
        tau, _ = kendalltau(np.abs(shap_vals[i]), np.abs(lime_vals[i]))
        scores[i] = tau if not np.isnan(tau) else 0.0
    return scores

## 4. Stability metrics

Two distinct notions, kept separate (v6 audit flagged that only one was implemented but
attributed to a citation describing the other):

- **Reproducibility**: re-run the explainer `STABILITY_N_RERUNS` times on the same instance,
  measure Jaccard similarity of the top-K feature set across runs. This is estimator variance.
- **Local robustness** (`calculate_max_sensitivity`, defined above): perturb the input slightly,
  measure how much the explanation changes. This is the quantity in Alvarez-Melis & Jaakkola
  (2018).


In [ ]:
def compute_reproducibility_jaccard(explain_fn, X_subset, k_top, n_reruns=10, seed=0):
    n = len(X_subset)
    jaccards = np.zeros(n)
    for i in range(n):
        x0 = X_subset[i:i + 1]
        top_sets = []
        for r in range(n_reruns):
            vals = explain_fn(x0, seed=seed + r)
            top = set(np.argsort(-np.abs(vals[0]))[:k_top])
            top_sets.append(top)
        pairs = [(a, b) for idx_a, a in enumerate(top_sets) for b in top_sets[idx_a + 1:]]
        scores = [len(a & b) / len(a | b) for a, b in pairs]
        jaccards[i] = np.mean(scores)
    return jaccards


## 5. Main training + explanation pipeline for M1

In [ ]:
def train_and_explain_m1(dataset_name):
    print(f"\n{'=' * 60}")
    print(f"M1 (XGBoost) -- {dataset_name}")
    print(f"{'=' * 60}")

    train_df = pd.read_parquet(f"{Config.INPUT_DIR}/{dataset_name}_train.parquet")
    test_df = pd.read_parquet(f"{Config.INPUT_DIR}/{dataset_name}_test_balanced.parquet")
    natural_df = pd.read_parquet(f"{Config.INPUT_DIR}/{dataset_name}_test_natural.parquet")

    y_train = train_df["TARGET"].values
    X_train = train_df.drop(columns=["TARGET"]).values
    feature_names = train_df.drop(columns=["TARGET"]).columns.tolist()

    y_test = test_df["TARGET"].values
    X_test = test_df.drop(columns=["TARGET"]).values

    y_nat = natural_df["TARGET"].values
    X_nat = natural_df.drop(columns=["TARGET"]).values

    # --- 1. Train XGBoost ---
    print("Training XGBoost...")
    model = xgb.XGBClassifier(
        max_depth=6, subsample=0.8, n_estimators=100,
        random_state=Config.SEED, eval_metric="logloss",
    )
    model.fit(X_train, y_train)
    joblib.dump(model, f"{Config.OUTPUT_DIR}/m1_xgb_{dataset_name}.pkl")
    print("Model saved.")

    def predict_fn(X):
        return model.predict_proba(X)[:, 1]

    # --- 2. Predictive performance + calibration, on both test sets ---
    for split_name, Xs, ys in [("balanced", X_test, y_test), ("natural", X_nat, y_nat)]:
        pred = predict_fn(Xs)
        auroc = roc_auc_score(ys, pred)
        auprc = average_precision_score(ys, pred)
        ece = compute_ece(ys, pred)
        nll = compute_nll(ys, pred)
        brier = compute_brier(ys, pred)

        metrics = {
            "predictive": {"auc": float(auroc), "auprc": float(auprc)},
            "calibration": {"ece": float(ece), "nll": float(nll), "brier": float(brier)},
            "n_samples": int(len(ys)), "prevalence": float(ys.mean()),
        }
        suffix = "" if split_name == "balanced" else "_natural"
        with open(f"{Config.OUTPUT_DIR}/m1_summary_metrics{suffix}_{dataset_name}.json", "w") as f:
            json.dump(metrics, f, indent=2)

        # M1 has no native uncertainty: epistemic_unc = 0 by construction (see notebook header).
        unc_df = pd.DataFrame({
            "sample_id": np.arange(len(ys)),
            "y_true": ys,
            "pred_mean": pred,
            "epistemic_unc": 0.0,
            "aleatoric_unc": -(pred * np.log(pred + 1e-12) + (1 - pred) * np.log(1 - pred + 1e-12)),
            "total_unc": -(pred * np.log(pred + 1e-12) + (1 - pred) * np.log(1 - pred + 1e-12)),
        })
        unc_df.to_csv(f"{Config.OUTPUT_DIR}/m1_uncertainty{suffix}_{dataset_name}.csv", index=False)

        print(f"  [{split_name:>8}] AUC={auroc:.4f}  AUPRC={auprc:.4f}  "
              f"ECE={ece:.4f}  NLL={nll:.4f}  Brier={brier:.4f}  "
              f"n={len(ys)}  prevalence={ys.mean():.4f}")

    base_probs = predict_fn(X_test)

    # Per-sample masking baseline drawn from the TRAIN distribution (v7 audit fix -- was
    # X_test.mean(axis=0), a single population-mean point; see calculate_faithfulness's
    # docstring). Fixed once per dataset, reused for every variant/method/K so all
    # comparisons in this notebook see the same baseline draw.
    baseline_rng = np.random.RandomState(Config.SEED)
    baseline_matrix = X_train[baseline_rng.choice(len(X_train), size=len(X_test), replace=True)]
    p_null_samples = predict_fn(baseline_matrix)  # saved to JSON below (v7 audit addition)

    # --- 3. TreeSHAP ---
    print("Computing TreeSHAP...")
    explainer_shap = shap.TreeExplainer(model)
    shap_vals = explainer_shap.shap_values(X_test)
    np.savez_compressed(f"{Config.OUTPUT_DIR}/m1_shap_{dataset_name}.npz", shap_values=shap_vals)

    # --- 4. LIME ---
    print("Computing LIME...")
    background_idx = np.random.RandomState(Config.SEED).choice(
        len(X_train), size=min(Config.LIME_BACKGROUND_SIZE, len(X_train)), replace=False
    )
    X_background = X_train[background_idx]
    explainer_lime = lime.lime_tabular.LimeTabularExplainer(
        X_background, feature_names=feature_names, class_names=["0", "1"],
        verbose=False, mode="classification",
    )
    lime_vals = np.zeros((len(X_test), len(feature_names)))
    for i in range(len(X_test)):
        exp = explainer_lime.explain_instance(
            X_test[i], model.predict_proba, num_features=len(feature_names),
            num_samples=Config.LIME_NSAMPLES,
        )
        for feat_idx, weight in exp.as_map()[1]:
            lime_vals[i, feat_idx] = weight
    np.savez_compressed(f"{Config.OUTPUT_DIR}/m1_lime_{dataset_name}.npz", lime_weights=lime_vals)

    # --- 5. Faithfulness, three variants x two methods ---
    k_list = Config.get_k_vals(X_test.shape[1])
    print(f"Faithfulness -- K grid: {k_list}  ({X_test.shape[1]} features)")
    # One masking pass per method yields all three variants -- see calculate_faithfulness's
    # docstring for why the old three-call form threw away two thirds of the work.
    faith_cache = {
        method_name: calculate_faithfulness(
            predict_fn, X_test, base_probs, attributions, k_list, baseline_matrix,
            variants=("prob", "logit", "norm"),
            random_repeats=Config.RANDOM_K_REPEATS, seed=Config.SEED,
            chunk_size=Config.FAITH_CHUNK,
        )
        for method_name, attributions in [("shap", shap_vals), ("lime", lime_vals)]
    }
    for variant in ["prob", "logit", "norm"]:
        for method_name in ["shap", "lime"]:
            comp, suff, comp_r, suff_r = faith_cache[method_name][variant]
            payload = {
                "M1": {
                    "comp_mean": {str(k): float(np.nanmean(comp[k])) for k in k_list},
                    "suff_mean": {str(k): float(np.nanmean(suff[k])) for k in k_list},
                    "comp_random_mean": {str(k): float(np.nanmean(comp_r[k])) for k in k_list},
                    "suff_random_mean": {str(k): float(np.nanmean(suff_r[k])) for k in k_list},
                    "comp_samples": {str(k): comp[k].tolist() for k in k_list},
                    "suff_samples": {str(k): suff[k].tolist() for k in k_list},
                    "comp_random_samples": {str(k): comp_r[k].tolist() for k in k_list},
                    "suff_random_samples": {str(k): suff_r[k].tolist() for k in k_list},
                    "k_list": k_list,
                    # v7 audit additions -- saved directly instead of reconstructed later:
                    "base_probs": base_probs.tolist(),
                    "p_null_samples": p_null_samples.tolist(),
                }
            }
            with open(f"{Config.OUTPUT_DIR}/faithfulness_{variant}_{method_name}_{dataset_name}.json", "w") as f:
                json.dump(payload, f)
            print(f"  [{variant:>5} / {method_name:>4}] "
                  f"comp@{k_list[len(k_list)//2]}={np.nanmean(comp[k_list[len(k_list)//2]]):.5f}")

    # --- 6. Explainer agreement (Kendall's W) ---
    kendall_scores = kendalls_w(shap_vals, lime_vals)
    print(f"Mean Kendall tau (SHAP vs LIME rankings): {np.mean(kendall_scores):.4f}")

    # --- 7. Stability: reproducibility + local robustness ---
    print("Computing stability metrics...")
    stab_idx = np.random.RandomState(Config.SEED).choice(
        len(X_test), size=min(Config.STABILITY_N_INSTANCES, len(X_test)), replace=False
    )
    X_stab = X_test[stab_idx]

    def shap_explain_fn(X, seed=0):
        return explainer_shap.shap_values(X)

    jac_shap = compute_reproducibility_jaccard(
        shap_explain_fn, X_stab, k_top=max(1, k_list[1]),
        n_reruns=Config.STABILITY_N_RERUNS, seed=Config.SEED,
    )

    def lime_explain_fn(X, seed=0):
        out = np.zeros((len(X), len(feature_names)))
        for i in range(len(X)):
            exp = explainer_lime.explain_instance(
                X[i], model.predict_proba, num_features=len(feature_names),
                num_samples=Config.LIME_NSAMPLES,
            )
            for feat_idx, weight in exp.as_map()[1]:
                out[i, feat_idx] = weight
        return out

    jac_lime = compute_reproducibility_jaccard(
        lime_explain_fn, X_stab, k_top=max(1, k_list[1]),
        n_reruns=Config.STABILITY_N_RERUNS, seed=Config.SEED,
    )

    max_sens_shap = calculate_max_sensitivity(
        shap_explain_fn, X_stab, n_perturbations=Config.SENSITIVITY_N_PERTURB,
        radius=Config.SENSITIVITY_RADIUS, seed=Config.SEED,
    )

    np.savez_compressed(
        f"{Config.OUTPUT_DIR}/m1_stability_{dataset_name}.npz",
        jaccard_shap=jac_shap, jaccard_lime=jac_lime, max_sensitivity_shap=max_sens_shap,
        stab_sample_idx=stab_idx,
    )
    print(f"  Reproducibility Jaccard: SHAP={np.mean(jac_shap):.4f}  LIME={np.mean(jac_lime):.4f}")
    print(f"  Local robustness (max-sensitivity, SHAP): {np.mean(max_sens_shap):.4f}")

    print(f"Finished {dataset_name}.")

## 6. Run for all datasets

In [ ]:
for ds in Config.DATASETS:
    try:
        train_and_explain_m1(ds)
    except Exception as e:
        import traceback
        print(f"FAILED on {ds}: {e}")
        traceback.print_exc()

print("\nAll datasets processed. Outputs are in /kaggle/working.")
print("Split into NB2-result (json/csv) and NB2-models (pkl) when downloading.")
